<a href="https://colab.research.google.com/github/Phavouredphavour/Pizza-Sales-Analysis-/blob/main/curriculum/phase-2b-sql/weeks-01-08-teaching/week-03-joins/02-thursday/exercises/week-03-thu-exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 3 — JOINs: LEFT JOIN, NULLs Across Tables & Three-Table Queries
## Phase 2b SQL | PORA Academy Cohort 7 — **Exercises** (Thursday)

Today you practise pulling an answer out of **more than one table at a time**. Every
question below is one real business question that no single Olist table can answer on
its own — you have to join.

Each question comes as **three cells**:

1. A **question** cell — the task in plain English, plus a hint about which tables to
   reach for and what to call your output columns.
2. A blank `%%sql qN <<` **answer cell** — replace `-- Your query here` with your query.
   The `qN <<` part captures the result into a pandas DataFrame named `qN`.
3. A **check cell** (plain Python) — run it straight after your query. A ✅ means your
   query behaved the way the question asked.

**Do not edit the check cells.** Some checks compare against a verified number; others
only check the *shape* of your answer (right columns, right number of rows, sensible
values) because the exact figure is for you to discover. A ✅ on a shape-only check
means your query is well formed — read the table it prints and interpret the numbers
yourself.

Alias your columns **exactly as the question asks** — the check cells look them up by name.

### Setup — run this cell first
It loads the eight Olist tables into a file-based SQLite database and connects the
`%%sql` magic to it. Because `autopandas` is on, every `%%sql` result comes back as a
pandas DataFrame, which is what lets the check cells inspect your answer. Run it once,
top of the notebook, before anything else.

In [1]:
# =====================================================================
# Olist SQL Setup — runs on BOTH Google Colab and a local machine.
# Run this cell FIRST. It loads the 8 Olist tables into a SQLite
# database and connects the %%sql magic to it. You should not need to
# edit anything unless auto-detection fails (see the two knobs below).
#
# Design notes:
# - We teach SQL with the %%sql cell magic (jupysql), not pd.read_sql().
# - jupysql opens its OWN connection, so the DB must be a real FILE
#   (a :memory: DB would be invisible to it).
# - We use jupysql (the maintained SQL magic). On Colab we install it,
#   because Colab ships the legacy ipython-sql, which (a) can't take a
#   connection by engine variable and (b) renders every result through
#   prettytable.__dict__[style], crashing on modern prettytable with
#   KeyError 'DEFAULT'/'SINGLE_BORDER'. jupysql fixes both.
# - autopandas=True makes every %%sql result a pandas DataFrame, which
#   lets the self-check cells assert on .iloc/.shape directly.
# =====================================================================
import os, glob, sqlite3, tempfile, zipfile
import pandas as pd

# --- Optional knobs (leave blank; only set if auto-detect fails) ------
LOCAL_DATA_DIR = ""   # local run: folder that holds olist_orders_dataset.csv
DRIVE_ZIP_PATH = ""   # Colab: full path to phase-2-python-sql.zip in your Drive
# ---------------------------------------------------------------------

# Detect Colab (google.colab only imports there). Outside Colab — including
# the content-pipeline validator — this falls through to the local branch.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ON_COLAB = True
except ModuleNotFoundError:
    ON_COLAB = False


def _colab_find_zip():
    """Locate phase-2-python-sql.zip in Drive WITHOUT a full recursive scan
    (globbing '/content/drive/MyDrive/**' walks the entire Drive over the
    network and can hang for many minutes). Try explicit paths first, then a
    depth- and count-bounded breadth-first search that prints progress."""
    if DRIVE_ZIP_PATH:
        if os.path.exists(DRIVE_ZIP_PATH):
            return DRIVE_ZIP_PATH
        raise FileNotFoundError(f"DRIVE_ZIP_PATH is set but not found: {DRIVE_ZIP_PATH}")

    target = "phase-2-python-sql.zip"
    # Fast, instant checks of the most likely spots (top of Drive + course folder).
    for cand in (
        f"/content/drive/MyDrive/{target}",
        f"/content/drive/MyDrive/Data Analysis and AI Automation Course Cohort 7/Dataset/{target}",
        f"/content/{target}",
    ):
        if os.path.exists(cand):
            return cand

    # Bounded BFS: depth <= 4, at most ~600 folders, skipping hidden dirs.
    print("Searching your Google Drive for phase-2-python-sql.zip ...")
    root, queue, scanned = "/content/drive/MyDrive", [("/content/drive/MyDrive", 0)], 0
    while queue:
        d, depth = queue.pop(0)
        hit = os.path.join(d, target)
        if os.path.exists(hit):
            return hit
        if depth >= 4:
            continue
        try:
            for e in os.scandir(d):
                if e.is_dir() and not e.name.startswith("."):
                    queue.append((e.path, depth + 1))
        except OSError:
            continue
        scanned += 1
        if scanned % 50 == 0:
            print(f"  ...scanned {scanned} folders")
        if scanned >= 600:
            break

    raise FileNotFoundError(
        "Could not quickly find phase-2-python-sql.zip in your Drive. Put the zip at the "
        "TOP of your Drive (My Drive) and re-run, or set DRIVE_ZIP_PATH at the top of this "
        "cell to its exact path.")


def _find_csv_dir():
    """Return the folder that actually contains olist_orders_dataset.csv."""
    roots = []
    env_dir = os.environ.get("OLIST_DATA_PATH", "")   # set by the pipeline validator
    if env_dir:
        roots.append(env_dir)
    if LOCAL_DATA_DIR:
        roots.append(LOCAL_DATA_DIR)

    if ON_COLAB:
        extract_path = "/content/olist_data"
        # unzip only the first time; reuse the extracted CSVs afterwards
        if not glob.glob(f"{extract_path}/**/olist_orders_dataset.csv", recursive=True):
            zip_path = _colab_find_zip()
            os.makedirs(extract_path, exist_ok=True)
            print(f"Unzipping {os.path.basename(zip_path)} ...")
            with zipfile.ZipFile(zip_path) as z:
                z.extractall(extract_path)
        roots.append(extract_path)
    else:
        # Local: search cwd (recursively) + a few common spots — never the whole
        # home dir (that recursive walk can be very slow). Set LOCAL_DATA_DIR if
        # your CSVs live elsewhere.
        roots += [os.getcwd(),
                  os.path.expanduser("~/Downloads"),
                  os.path.expanduser("~/Desktop"),
                  os.path.expanduser("~/olist")]

    for root in roots:
        if os.path.exists(os.path.join(root, "olist_orders_dataset.csv")):
            return root
        hits = glob.glob(os.path.join(root, "**", "olist_orders_dataset.csv"), recursive=True)
        if hits:
            return os.path.dirname(hits[0])

    raise FileNotFoundError(
        "Olist CSVs not found. Set LOCAL_DATA_DIR (local) or DRIVE_ZIP_PATH (Colab) at "
        "the top of this cell.")


DATA_DIR = _find_csv_dir()
print("Data folder:", DATA_DIR)

# Build a file-based SQLite DB shared by pandas (loading) and jupysql (querying).
DB_PATH = os.environ.get("OLIST_DB_PATH") or (
    "/content/olist.db" if ON_COLAB else os.path.join(tempfile.gettempdir(), "olist.db"))

tables = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "product_category_translation": "product_category_name_translation.csv",
}

conn = sqlite3.connect(DB_PATH)
for table_name, filename in tables.items():
    df = pd.read_csv(os.path.join(DATA_DIR, filename))
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    print(f"Loaded {table_name}: {len(df):,} rows")
conn.close()
print("\nDatabase ready.")

# On Colab, install jupysql so `%load_ext sql` loads it instead of the legacy
# ipython-sql (see header). Off Colab (local / pipeline validator) jupysql is
# already installed, so we skip the install and stay offline-safe.
if ON_COLAB:
    get_ipython().run_line_magic("pip", "install --quiet --upgrade jupysql")

get_ipython().run_line_magic("load_ext", "sql")

# Guard: if the legacy ipython-sql was already loaded earlier THIS session (e.g.
# an older cell ran first), the freshly installed jupysql cannot hot-swap in — a
# runtime restart is the only fix. jupysql exposes sql.connection.ConnectionManager;
# ipython-sql does not. Stop with a clear instruction instead of a later cryptic
# prettytable KeyError.
import sql.connection as _sqlconn
if not hasattr(_sqlconn, "ConnectionManager"):
    raise RuntimeError(
        "Legacy ipython-sql is active, not jupysql. On Colab: Runtime -> Restart session, "
        "then run THIS setup cell first (before any other cell). Locally: "
        "pip install --upgrade jupysql and restart the kernel."
    )

# Connect the %%sql magic to the SAME database file. autopandas=True is REQUIRED
# (see header). We connect with run_line_magic (not a literal `%sql` line) so the
# computed DB_PATH is interpolated correctly. Do NOT set SqlMagic.style.
get_ipython().run_line_magic("config", "SqlMagic.autopandas = True")
get_ipython().run_line_magic("config", "SqlMagic.feedback = 0")
get_ipython().run_line_magic("sql", f"sqlite:///{DB_PATH}")

# Verify (expected row counts — do not alter without re-running against data):
#   orders 99,441 | customers 99,441 | order_items 112,650 | order_payments 103,886
#   order_reviews 99,224 | products 32,951 | sellers 3,095 | product_category_translation 71

Mounted at /content/drive
Searching your Google Drive for phase-2-python-sql.zip ...
Unzipping phase-2-python-sql.zip ...
Data folder: /content/olist_data/phase-2-python-sql
Loaded orders: 99,441 rows
Loaded customers: 99,441 rows
Loaded order_items: 112,650 rows
Loaded order_payments: 103,886 rows
Loaded order_reviews: 99,224 rows
Loaded products: 32,951 rows
Loaded sellers: 3,095 rows
Loaded product_category_translation: 71 rows

Database ready.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.8/192.8 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 607.8/607.8 kB 23.4 MB/s eta 0:00:00


## Question 1 — Delivered orders from Minas Gerais

The operations team wants a state-by-state delivery report and is starting with **Minas
Gerais (`MG`)**. The problem: `orders` knows the *status* of an order but not the
customer's state, and `customers` knows the *state* but not the status. One table cannot
answer this.

Join `orders` to `customers` on `customer_id`, keep only rows where the status is
`delivered` **and** the customer state is `MG`, and count them.

**Hint:** an `INNER JOIN` is right here — an order with no matching customer would be
meaningless. Alias your count as `mg_delivered`.

**Expected:** 11,354 delivered orders

In [4]:
%%sql q1 <<
-- Your query here
SELECT COUNT(*) AS mg_delivered
FROM orders o
JOIN customers c
  ON o.customer_id = c.customer_id
WHERE o.order_status = 'delivered' AND c.customer_state = 'MG'

In [5]:
# --- CHECK Q1 — do not edit ---
assert "mg_delivered" in q1.columns, "Q1: alias your count as mg_delivered"
assert int(q1.iloc[0]["mg_delivered"]) == 11354, (
    f"Q1: expected 11,354 delivered MG orders, got {int(q1.iloc[0]['mg_delivered']):,}"
)
print("✅ Q1 correct")
q1  # show the result of your query

✅ Q1 correct


,mg_delivered
0,11354


## Question 2 — Average payment value for São Paulo customers

Finance wants to know what a typical São Paulo (`SP`) customer actually pays. The payment
amount lives in `order_payments`, the state lives in `customers`, and `orders` is the
bridge between them — so this is a **three-table join**: `orders` → `customers` →
`order_payments`.

Filter to customers in `SP` and return the average payment value, rounded to 2 decimal
places.

**Hint:** join `orders` to `customers` on `customer_id`, then `orders` to
`order_payments` on `order_id`. Alias the result as `avg_payment`. Use
`ROUND(AVG(...), 2)`.

**Note:** an order can be split across several payment rows (e.g. two vouchers plus a
card), so this is the average *per payment row*, not per order. Say what you are
measuring when you report a number like this.

**Expected:** a single row with one column, `avg_payment` — a sensible Brazilian-Real
amount. Read the value your query returns and note it down; the check below verifies the
shape of your answer, not a pre-supplied figure.

In [14]:
%%sql q2 <<
-- Your query here
SELECT ROUND(AVG(op.payment_value), 2) AS avg_payment
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN order_payments op ON o.order_id = op.order_id
WHERE c.customer_state = 'SP'

In [15]:
# --- CHECK Q2 — do not edit (structural check: shape, not a fixed value) ---
assert q2.shape[0] == 1, f"Q2: expected exactly 1 row, got {q2.shape[0]}"
assert "avg_payment" in q2.columns, "Q2: alias your result as avg_payment"
_v = float(q2.iloc[0]["avg_payment"])
assert _v > 0, "Q2: an average payment value must be positive"
assert 10 < _v < 1000, (
    f"Q2: {_v} is outside a sensible range for an average payment — "
    "check your join keys and your WHERE filter"
)
assert round(_v, 2) == _v, "Q2: round your average to 2 decimal places with ROUND(..., 2)"
print(f"✅ Q2 well formed — average payment for SP customers: R$ {_v:,.2f}")
q2  # show the result of your query

✅ Q2 well formed — average payment for SP customers: R$ 137.50


,avg_payment
0,137.5


## Question 3 — Which states have a real seller base?

Before Olist invests in regional seller support, it needs to know where its sellers
actually are. Count the sellers in each state, but show only the states that clear a
meaningful bar: **50 or more sellers**. Order the result from most sellers to fewest.

**Hint:** this one needs no join at all — `sellers` already holds `seller_state`. It is
here to remind you that a join is a tool, not a reflex: reach for one only when the
answer genuinely spans two tables. You will need `GROUP BY` and `HAVING` (Week 2) — the
filter is on the *count*, not on individual rows, so it cannot go in `WHERE`.

Alias your count as `seller_count` and keep the state column named `seller_state`.

**Expected:** one row per qualifying state, every `seller_count` at or above 50, sorted
descending. The check verifies that shape — read the table to see which states made the
cut.

In [19]:
%%sql q3 <<
-- Your query here
SELECT s.seller_state,
      COUNT(*) AS seller_count
FROM sellers s
GROUP BY s.seller_state
HAVING seller_count >= 50
ORDER BY seller_count DESC

In [20]:
# --- CHECK Q3 — do not edit (structural check: shape, not a fixed value) ---
for _c in ("seller_state", "seller_count"):
    assert _c in q3.columns, f"Q3: your result needs a column named {_c}"
assert q3.shape[0] >= 1, "Q3: no rows returned — did the HAVING filter run before GROUP BY?"
assert q3.shape[0] <= 27, (
    f"Q3: {q3.shape[0]} rows — Brazil has 27 states, so you are returning too many "
    "(are you grouping by state?)"
)
_counts = [int(x) for x in q3["seller_count"]]
assert min(_counts) >= 50, (
    f"Q3: smallest seller_count is {min(_counts)} — HAVING should drop states below 50"
)
assert _counts == sorted(_counts, reverse=True), (
    "Q3: rows are not sorted from most sellers to fewest — add ORDER BY seller_count DESC"
)
assert q3["seller_state"].nunique() == q3.shape[0], "Q3: each state should appear once"
print(f"✅ Q3 well formed — {q3.shape[0]} states have 50+ sellers")
q3  # show the result of your query

✅ Q3 well formed — 6 states have 50+ sellers


,seller_state,seller_count
0,SP,1849
1,PR,349
2,MG,244
3,SC,190
4,RJ,171
5,RS,129


## Question 4 — Reviewed vs unreviewed orders (open-ended)

**Think before you type.** Do orders that received a review have different average item
prices than orders that received none?

The trap: if you write `order_items JOIN order_reviews`, you have already thrown away
every unreviewed order — an `INNER JOIN` keeps only rows that match on *both* sides. To
see the orders with **no** review at all, you need a `LEFT JOIN` from `order_items` to
`order_reviews` and then a test on whether the right-hand side came back `NULL`
(`r.order_id IS NULL` means "no review exists"). Remember: `IS NULL`, never `= NULL`.

**Your task:** write one query that reports the average item price for each of the two
groups — orders **with** a review and orders **without** one — so the comparison is
visible in a single table.

**A sketch of one approach** (there are others, and yours may look different):

```
SELECT CASE WHEN r.order_id IS NULL THEN 'no review' ELSE 'has review' END AS review_group,
       COUNT(DISTINCT oi.order_id) AS order_count,
       ROUND(AVG(oi.price), 2)     AS avg_item_price
FROM   order_items oi
LEFT JOIN order_reviews r ON ...
GROUP BY review_group
```

**Then answer in words** (write it in the markdown cell after the check, or say it out
loud to your partner):
1. Which group has the higher average item price?
2. Is the gap big enough to act on, or is one group so small that the average is noisy?
3. Why would an `INNER JOIN` have made this question impossible to answer?

**Expected:** there is no single "right number" here — the check below only confirms your
query ran and returned rows. The reasoning is the exercise.

In [22]:
%%sql q4 <<
-- Your query here
SELECT CASE WHEN
                r.order_id IS NULL THEN 'no review'
                                  ELSE 'has review'
                END                           AS review_group,
                COUNT(DISTINCT oi.order_id)   AS order_items_count,
                ROUND(AVG(oi.price), 2)       AS avg_item_price
FROM order_items oi
LEFT JOIN order_reviews r ON oi.order_id = r.order_id
GROUP BY review_group

In [23]:
# --- CHECK Q4 — do not edit (open-ended: runs your query, does not grade the number) ---
assert hasattr(q4, "shape"), "Q4: q4 should be a DataFrame — use the %%sql q4 << capture syntax"
assert q4.shape[0] >= 1, "Q4: your query returned no rows — check your LEFT JOIN and GROUP BY"
assert q4.shape[1] >= 2, (
    "Q4: return at least two columns — the group label and the average item price — "
    "so the two groups can be compared side by side"
)
print(f"✅ Q4 query ran — {q4.shape[0]} group(s) returned. Now interpret it:")
print("   • Which group has the higher average item price?")
print("   • Is the difference large enough to act on?")
print("   • What would an INNER JOIN have hidden from you?")
q4  # show the result of your query

✅ Q4 query ran — 2 group(s) returned. Now interpret it:
   • Which group has the higher average item price?
   • Is the difference large enough to act on?
   • What would an INNER JOIN have hidden from you?


,review_group,order_items_count,avg_item_price
0,has review,97917,120.38
1,no review,749,132.38


## Question 5 — Do cheaper items get worse reviews?

Customer experience suspects that price shapes satisfaction. Test it: for each review
score from 1 to 5, report how many orders carry that score and the average price of the
items in those orders.

Join `order_items` to `order_reviews` on `order_id`, group by `review_score`, and order
the result by score so the table reads 1 → 5.

**Hint:** name your columns `review_score`, `order_count`, and `avg_item_price`. Because
an order can have several items *and* several review rows, count orders with
`COUNT(DISTINCT oi.order_id)` rather than `COUNT(*)` — otherwise the same order is
counted once per item. Round the average price to 2 decimal places.

**Expected:** five rows, one per score from 1 to 5, each with a positive average item
price. The check verifies that shape — the pattern in the numbers is yours to read and
explain.

In [26]:
%%sql q5 <<
-- Your query here
SELECT review_score,
      COUNT(DISTINCT oi.order_id)                AS order_count,
      ROUND(AVG(oi.price), 2)                    AS avg_item_price
FROM order_items oi
JOIN order_reviews r ON oi.order_id = r.order_id
GROUP BY review_score
ORDER BY review_score

In [27]:
# --- CHECK Q5 — do not edit (structural check: shape, not a fixed value) ---
for _c in ("review_score", "order_count", "avg_item_price"):
    assert _c in q5.columns, f"Q5: your result needs a column named {_c}"
assert q5.shape[0] == 5, (
    f"Q5: expected 5 rows (one per review score 1-5), got {q5.shape[0]} — "
    "did you GROUP BY review_score?"
)
_scores = [int(x) for x in q5["review_score"]]
assert _scores == [1, 2, 3, 4, 5], (
    f"Q5: expected review scores 1,2,3,4,5 in ascending order, got {_scores} — "
    "add ORDER BY review_score"
)
_prices = [float(x) for x in q5["avg_item_price"]]
assert all(p > 0 for p in _prices), "Q5: every average item price should be positive"
assert all(round(p, 2) == p for p in _prices), "Q5: round avg_item_price to 2 decimal places"
_orders = [int(x) for x in q5["order_count"]]
assert all(o > 0 for o in _orders), "Q5: every score should cover at least one order"
assert sum(_orders) < 120000, (
    "Q5: your order counts add up to more orders than Olist has — "
    "use COUNT(DISTINCT oi.order_id), not COUNT(*)"
)
print("✅ Q5 well formed — now read the pattern:")
print("   Is avg_item_price higher for 5-star orders than for 1-star orders?")
q5  # show the result of your query

✅ Q5 well formed — now read the pattern:
   Is avg_item_price higher for 5-star orders than for 1-star orders?


,review_score,order_count,avg_item_price
0,1,10854,127.35
1,2,3086,115.85
2,3,8107,110.06
3,4,19065,118.60
4,5,57006,121.22
